# SARSA (State-Action-Reward-State-Action)

## Learning Objectives
1. Implement on-policy SARSA on GridWorld and contrast with Q-learning's off-policy behavior
2. Build Expected SARSA and measure variance reduction vs standard SARSA
3. Demonstrate SARSA's safe-path behavior on Cliff Walking
4. Implement SARSA(lambda) with eligibility traces and sweep lambda values

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
print('Libraries loaded')
print(f'NumPy: {np.__version__}')

## Level 1: Basic SARSA on GridWorld

On-policy TD control: Q(s,a) += alpha * [r + gamma * Q(s',a') - Q(s,a)]  
a' is sampled from the SAME epsilon-greedy policy — key difference from Q-learning.

In [ ]:
class GridWorld:
    """4x4 GridWorld: start=(0,0), goal=(3,3), wall=(1,1)."""

    def __init__(self, size: int = 4):
        self.size = size
        self.goal = (size - 1, size - 1)
        self.wall = (1, 1)
        self.n_actions = 4
        self.n_states = size * size
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.reset()

    def reset(self) -> int:
        self.pos = (0, 0)
        return self._idx(self.pos)

    def _idx(self, pos) -> int:
        return pos[0] * self.size + pos[1]

    def step(self, action: int):
        dr, dc = self.action_deltas[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc
        if not (0 <= nr < self.size and 0 <= nc < self.size):
            return self._idx(self.pos), -1.0, False
        npos = (nr, nc)
        if npos == self.wall:
            return self._idx(self.pos), -5.0, False
        self.pos = npos
        if self.pos == self.goal:
            return self._idx(self.pos), 10.0, True
        return self._idx(self.pos), -1.0, False


def epsilon_greedy(Q_row: np.ndarray, epsilon: float) -> int:
    """Select action with epsilon-greedy policy."""
    if np.random.rand() < epsilon:
        return np.random.randint(len(Q_row))
    return int(np.argmax(Q_row))


def run_sarsa(
    env: GridWorld,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
) -> tuple:
    """Tabular SARSA. Returns (Q_table, rewards_per_episode)."""
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []

    for ep in range(n_episodes):
        state = env.reset()
        # SARSA: select FIRST action BEFORE the loop (on-policy)
        action = epsilon_greedy(Q[state], epsilon)
        total_reward = 0.0
        done = False
        steps = 0

        while not done and steps < 200:
            next_state, reward, done = env.step(action)
            # Sample next action from behavior policy (on-policy — KEY difference)
            next_action = epsilon_greedy(Q[next_state], epsilon)

            # SARSA update: uses Q(s', a') where a' is from behavior policy
            td_target = reward + gamma * Q[next_state, next_action] * (1 - done)
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error

            total_reward += reward
            state, action = next_state, next_action  # Advance both s and a
            steps += 1

        rewards.append(total_reward)

    return Q, rewards


def run_q_learning(
    env: GridWorld,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
) -> tuple:
    """Q-learning baseline for comparison."""
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        steps = 0
        while not done and steps < 200:
            action = epsilon_greedy(Q[state], epsilon)
            ns, r, done = env.step(action)
            Q[state, action] += alpha * (r + gamma * np.max(Q[ns]) * (1 - done) - Q[state, action])
            total_reward += r; state = ns; steps += 1
        rewards.append(total_reward)
    return Q, rewards


env = GridWorld()
# Compare SARSA with different epsilon values
for eps in [0.1, 0.3]:
    np.random.seed(42)
    Q_s, r_s = run_sarsa(env, n_episodes=500, epsilon=eps)
    print(f'SARSA eps={eps}: mean reward (last 50) = {np.mean(r_s[-50:]):.2f}')

np.random.seed(42)
Q_sarsa, rewards_sarsa = run_sarsa(env)
action_names = ['Up', 'Down', 'Left', 'Right']
policy = np.array([action_names[np.argmax(Q_sarsa[s])] for s in range(16)]).reshape(4, 4)
print('\nSARSA Greedy Policy:')
for row in policy:
    print('  '.join(f'{a:>5}' for a in row))

## Level 2: Expected SARSA

Replace Q(s',a') with E_{pi}[Q(s',a')] = sum_a pi(a|s') * Q(s',a').  
Eliminates variance from action sampling at the cost of O(|A|) computation per step.

In [ ]:
def expected_value_under_epsilon_greedy(Q_row: np.ndarray, epsilon: float) -> float:
    """Compute E_{pi}[Q(s,a)] under epsilon-greedy policy."""
    n_actions = len(Q_row)
    greedy_action = int(np.argmax(Q_row))
    # Each action has probability epsilon/n_actions of being chosen randomly
    # Greedy action gets an extra (1 - epsilon) probability
    probs = np.full(n_actions, epsilon / n_actions)
    probs[greedy_action] += (1.0 - epsilon)
    return float(np.dot(probs, Q_row))


def run_expected_sarsa(
    env: GridWorld,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
) -> tuple:
    """Expected SARSA: replaces sampled Q(s',a') with expectation over all actions."""
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []
    per_step_td_errors = []

    for ep in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(Q[state], epsilon)
        total_reward = 0.0
        ep_td = []
        done = False
        steps = 0

        while not done and steps < 200:
            next_state, reward, done = env.step(action)

            # Expected value under epsilon-greedy
            expected_q_next = expected_value_under_epsilon_greedy(Q[next_state], epsilon) * (1 - done)

            # Expected SARSA update: use expectation instead of sample
            td_error = reward + gamma * expected_q_next - Q[state, action]
            Q[state, action] += alpha * td_error

            ep_td.append(abs(td_error))
            total_reward += reward
            # Still sample next action for actual behavior (but don't use it for update)
            action = epsilon_greedy(Q[next_state], epsilon)
            state = next_state
            steps += 1

        rewards.append(total_reward)
        per_step_td_errors.append(np.mean(ep_td) if ep_td else 0.0)

    return Q, rewards, per_step_td_errors


# Run comparison across multiple seeds for reliable variance measurement
n_seeds = 5
sarsa_rewards_all = []
exp_sarsa_rewards_all = []

for seed in range(n_seeds):
    np.random.seed(seed)
    _, r_s, _ = run_sarsa(env, n_episodes=300)
    sarsa_rewards_all.append(r_s)

    np.random.seed(seed)
    _, r_e, _ = run_expected_sarsa(env, n_episodes=300)
    exp_sarsa_rewards_all.append(r_e)

sarsa_arr = np.array(sarsa_rewards_all)
exp_arr = np.array(exp_sarsa_rewards_all)

print('Variance comparison (last 50 episodes, averaged over 5 seeds):')
print(f'SARSA         - mean: {sarsa_arr[:, -50:].mean():.2f}, std across runs: {sarsa_arr[:, -50:].mean(axis=1).std():.2f}')
print(f'Expected SARSA - mean: {exp_arr[:, -50:].mean():.2f}, std across runs: {exp_arr[:, -50:].mean(axis=1).std():.2f}')
print('Expected SARSA should show lower variance (more consistent performance across seeds)')

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
mean_s = sarsa_arr.mean(axis=0)
std_s = sarsa_arr.std(axis=0)
mean_e = exp_arr.mean(axis=0)
std_e = exp_arr.std(axis=0)

episodes = np.arange(300)
ax.plot(episodes, mean_s, label='SARSA', color='blue')
ax.fill_between(episodes, mean_s - std_s, mean_s + std_s, alpha=0.2, color='blue')
ax.plot(episodes, mean_e, label='Expected SARSA', color='orange')
ax.fill_between(episodes, mean_e - std_e, mean_e + std_e, alpha=0.2, color='orange')

ax.set_title('SARSA vs Expected SARSA: Mean +/- Std over 5 Seeds')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/sarsa_expected.png', dpi=80, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/sarsa_expected.png')

## Real-World Example 1: SARSA on Cliff Walking (Safety Demonstration)

SARSA's on-policy values account for epsilon-greedy exploration near the cliff.  
The agent learns to take the safe high road rather than the risky cliff edge.

In [ ]:
class CliffWorld:
    """4 rows x 12 cols. Cliff at row 3, cols 1-10. Start=(3,0). Goal=(3,11)."""

    def __init__(self):
        self.rows = 4; self.cols = 12
        self.start = (3, 0); self.goal = (3, 11)
        self.cliff = {(3, c) for c in range(1, 11)}
        self.n_states = self.rows * self.cols
        self.n_actions = 4
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.reset()

    def reset(self):
        self.pos = self.start
        return self._idx(self.pos)

    def _idx(self, pos):
        return pos[0] * self.cols + pos[1]

    def step(self, action: int):
        dr, dc = self.action_deltas[action]
        r = max(0, min(self.rows - 1, self.pos[0] + dr))
        c = max(0, min(self.cols - 1, self.pos[1] + dc))
        self.pos = (r, c)
        if self.pos in self.cliff:
            self.pos = self.start
            return self._idx(self.pos), -100.0, False
        if self.pos == self.goal:
            return self._idx(self.pos), 0.0, True
        return self._idx(self.pos), -1.0, False


def run_cliff_sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """SARSA on cliff world."""
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []
    for ep in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(Q[state], epsilon)
        total = 0.0; done = False; steps = 0
        while not done and steps < 500:
            ns, r, done = env.step(action)
            next_action = epsilon_greedy(Q[ns], epsilon)
            Q[state, action] += alpha * (r + gamma * Q[ns, next_action] * (1-done) - Q[state, action])
            total += r; state, action = ns, next_action; steps += 1
        rewards.append(total)
    return Q, rewards


def run_cliff_qlearning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """Q-learning on cliff world."""
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []
    for ep in range(n_episodes):
        state = env.reset()
        total = 0.0; done = False; steps = 0
        while not done and steps < 500:
            action = epsilon_greedy(Q[state], epsilon)
            ns, r, done = env.step(action)
            Q[state, action] += alpha * (r + gamma * np.max(Q[ns]) * (1-done) - Q[state, action])
            total += r; state = ns; steps += 1
        rewards.append(total)
    return Q, rewards


cliff_env = CliffWorld()
np.random.seed(42)
Q_sarsa_cliff, sarsa_cliff_r = run_cliff_sarsa(cliff_env, n_episodes=500)
np.random.seed(42)
Q_ql_cliff, ql_cliff_r = run_cliff_qlearning(cliff_env, n_episodes=500)

print(f'SARSA on Cliff (epsilon=0.1) training reward (last 100): {np.mean(sarsa_cliff_r[-100:]):.1f}')
print(f'Q-learning training reward (last 100): {np.mean(ql_cliff_r[-100:]):.1f}')
print('SARSA higher during training (avoids cliff during exploration)')
print('Q-learning lower during training (walks close to cliff; random moves fall off)')

## Real-World Example 2: Windy GridWorld (Stochastic Environment)

Stochastic wind pushes the agent up by 1-2 rows with probability 0.3.  
SARSA accounts for wind stochasticity in its value estimates; Q-learning ignores it.

In [ ]:
class WindyGridWorld:
    """4x4 GridWorld with stochastic wind. Wind pushes agent up with probability wind_prob."""

    def __init__(self, size: int = 4, wind_prob: float = 0.3):
        self.size = size
        self.wind_prob = wind_prob
        self.goal = (size - 1, size - 1)
        self.n_actions = 4
        self.n_states = size * size
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.reset()

    def reset(self):
        self.pos = (0, 0)
        return self._idx(self.pos)

    def _idx(self, pos):
        return pos[0] * self.size + pos[1]

    def step(self, action: int):
        dr, dc = self.action_deltas[action]
        r, c = self.pos[0] + dr, self.pos[1] + dc
        # Stochastic wind: push up (decrease row) with wind_prob
        if np.random.rand() < self.wind_prob:
            r -= 1  # Wind blows agent upward
        # Clamp to grid boundaries
        r = max(0, min(self.size - 1, r))
        c = max(0, min(self.size - 1, c))
        self.pos = (r, c)
        if self.pos == self.goal:
            return self._idx(self.pos), 10.0, True
        return self._idx(self.pos), -1.0, False


def run_sarsa_generic(env, Q_init, n_episodes, alpha, gamma, epsilon, use_sarsa=True):
    """Generic SARSA or Q-learning runner for comparison."""
    Q = Q_init.copy()
    rewards = []
    for ep in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(Q[state], epsilon)
        total = 0.0; done = False; steps = 0
        while not done and steps < 300:
            ns, r, done = env.step(action)
            if use_sarsa:
                na = epsilon_greedy(Q[ns], epsilon)
                target = r + gamma * Q[ns, na] * (1 - done)
            else:
                target = r + gamma * np.max(Q[ns]) * (1 - done)
                na = epsilon_greedy(Q[ns], epsilon)
            Q[state, action] += alpha * (target - Q[state, action])
            total += r; state, action = ns, na; steps += 1
        rewards.append(total)
    return Q, rewards


windy = WindyGridWorld(size=4, wind_prob=0.3)
Q_init = np.zeros((16, 4))

sarsa_windy_rewards = []
ql_windy_rewards = []

for seed in range(10):
    np.random.seed(seed)
    _, rw_s = run_sarsa_generic(windy, Q_init, 500, 0.1, 0.95, 0.1, use_sarsa=True)
    sarsa_windy_rewards.append(rw_s)
    np.random.seed(seed)
    _, rw_q = run_sarsa_generic(windy, Q_init, 500, 0.1, 0.95, 0.1, use_sarsa=False)
    ql_windy_rewards.append(rw_q)

sarsa_w_arr = np.array(sarsa_windy_rewards)
ql_w_arr = np.array(ql_windy_rewards)

print('Windy GridWorld (10 seeds, last 50 episodes):')
print(f'SARSA      mean={sarsa_w_arr[:,-50:].mean():.2f}, std={sarsa_w_arr[:,-50:].mean(axis=1).std():.2f}')
print(f'Q-learning mean={ql_w_arr[:,-50:].mean():.2f}, std={ql_w_arr[:,-50:].mean(axis=1).std():.2f}')
print('SARSA generally better in stochastic environments during training')

## Real-World Example 3: SARSA(lambda) — Eligibility Traces for On-Policy Control

SARSA(lambda) propagates TD errors backward to all recently visited (s,a) pairs.  
Compare lambda in {0, 0.5, 0.9} on convergence speed.

In [ ]:
def run_sarsa_lambda(
    env: GridWorld,
    lam: float = 0.9,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
) -> list:
    """SARSA(lambda) with replacing eligibility traces. On-policy credit assignment."""
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []

    for ep in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(Q[state], epsilon)
        # Eligibility trace: one value per (state, action)
        E = np.zeros((env.n_states, env.n_actions))
        total_reward = 0.0
        done = False
        steps = 0

        while not done and steps < 300:
            next_state, reward, done = env.step(action)
            next_action = epsilon_greedy(Q[next_state], epsilon)

            # SARSA TD error using actual next action (on-policy)
            td_error = reward + gamma * Q[next_state, next_action] * (1 - done) - Q[state, action]

            # Replacing trace: cap at 1 to prevent unbounded accumulation
            E[state, action] = 1.0

            # Update all Q values proportional to eligibility
            Q += alpha * td_error * E

            # Decay all traces by gamma * lambda (eligibility fades over time)
            E *= gamma * lam

            total_reward += reward
            state, action = next_state, next_action
            steps += 1

        episode_rewards.append(total_reward)

    return episode_rewards


env = GridWorld()
lambdas = [0.0, 0.5, 0.9]
colors = ['steelblue', 'darkorange', 'green']
sarsa_lambda_rewards = {}

for lam in lambdas:
    results = []
    for seed in range(5):
        np.random.seed(seed)
        results.append(run_sarsa_lambda(env, lam=lam, n_episodes=400))
    sarsa_lambda_rewards[lam] = np.array(results)
    mean_r = sarsa_lambda_rewards[lam][:, -50:].mean()
    print(f'SARSA(lambda={lam:.1f}): mean reward (last 50) = {mean_r:.2f}')

## Comparison: Q-Learning vs SARSA vs Expected SARSA

Safety vs optimality tradeoff on cliff walking.  
Also plot SARSA(lambda) convergence sweep.

In [ ]:
def smooth(arr, w=20):
    """Moving average smoothing."""
    return np.convolve(arr, np.ones(w) / w, mode='valid')


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Cliff walking reward curves ---
ax = axes[0]
ax.plot(smooth(sarsa_cliff_r, 20), color='blue', label='SARSA (safe path)')
ax.plot(smooth(ql_cliff_r, 20), color='red', label='Q-Learning (risky path)')
ax.set_title('Cliff Walking: Training Performance')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward (smoothed)')
ax.set_ylim(-250, 10)
ax.legend()
ax.axhline(-13, color='blue', linestyle='--', alpha=0.4, label='SARSA optimal (-13)')
ax.axhline(-11, color='red', linestyle='--', alpha=0.4, label='QL optimal (-11)')

# --- Plot 2: SARSA(lambda) convergence sweep ---
ax = axes[1]
for lam, col in zip(lambdas, colors):
    mean_curve = sarsa_lambda_rewards[lam].mean(axis=0)
    std_curve = sarsa_lambda_rewards[lam].std(axis=0)
    smoothed_mean = smooth(mean_curve, 15)
    ax.plot(smoothed_mean, color=col, label=f'lambda={lam:.1f}')
ax.set_title('SARSA(lambda): Convergence Speed')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward (smoothed)')
ax.legend()

# --- Plot 3: SARSA vs Expected SARSA final performance distribution ---
ax = axes[2]
sarsa_final = [np.mean(r[-50:]) for r in sarsa_rewards_all]
exp_sarsa_final = [np.mean(r[-50:]) for r in exp_sarsa_rewards_all]
ax.boxplot([sarsa_final, exp_sarsa_final], labels=['SARSA', 'Expected SARSA'])
ax.set_title('Final Performance Distribution (5 seeds, last 50 eps)')
ax.set_ylabel('Mean Reward')

plt.tight_layout()
plt.savefig('/tmp/sarsa_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print('Comparison plot saved to /tmp/sarsa_comparison.png')

# Summary table
print('\n=== Algorithm Comparison Summary ===')
print(f'{"Algorithm":<25} {"Training Reward":<20} {"Policy Type":<20}')
print('-' * 65)
print(f'{"SARSA":<25} {np.mean(sarsa_cliff_r[-100:]):<20.1f} {"On-policy (safe)":<20}')
print(f'{"Q-Learning":<25} {np.mean(ql_cliff_r[-100:]):<20.1f} {"Off-policy (optimal)":<20}')
print(f'{"Expected SARSA":<25} {exp_arr[:,-50:].mean():<20.2f} {"On-policy (low var)":<20}')

## Key Takeaways

**Core idea:** SARSA is on-policy TD control — it learns Q-values for the actual behavior policy (including exploration noise). This makes it safer in risky environments but prevents learning the globally optimal greedy policy unless epsilon decays to 0.

**Variants and when to use:**

| Method | Use when | Trade-off |
|--------|----------|----------|
| SARSA | Safe exploration needed; training = deployment | Cannot learn off-policy; slightly suboptimal |
| Expected SARSA | Low variance needed; small action space | O(|A|) compute per step; otherwise free |
| SARSA(lambda) | Sparse rewards; long episodes | Higher memory; unstable with noisy rewards |
| Q-learning | Offline data reuse; optimal policy needed | Overestimates values; ignores exploration risk |

**Common failure modes:**
- Using greedy next action instead of sampled: silent Q-learning bug; SARSA safety property lost
- High lambda with noisy rewards: trace amplifies noise; Q-values oscillate
- Not decaying epsilon to 0: final policy is epsilon-optimal, not globally optimal

**Related concepts:**
- [06-q-learning](./06-q-learning.ipynb) — off-policy counterpart
- [09-policy-gradient](./09-policy-gradient.ipynb) — directly parameterizes on-policy behavior
- [08-deep-q-networks](./08-deep-q-networks.ipynb) — neural Q-learning for large state spaces

## Exercises

1. **Bug hunt:** Modify SARSA to accidentally use `np.argmax(Q[next_state])` instead of sampling `next_action`. Confirm it now behaves like Q-learning on cliff walking.
2. **Epsilon decay:** Add ε decay (1.0 → 0.01 over 500 episodes) to SARSA on cliff walking. Does the final greedy policy match Q-learning's optimal path?
3. **Expected SARSA analysis:** Profile the wall-clock time per episode for SARSA vs Expected SARSA as `|A|` grows from 4 to 16. At what `|A|` does the overhead become significant?
4. **Lambda sweep:** On a 6×6 GridWorld with sparse reward (only at goal), sweep lambda from 0 to 1.0 in steps of 0.1. Plot mean steps-to-goal vs lambda.